# Sparsity: Systematic Study

**Δ from baseline** (one parameter):
```
+ topk_neurons = ratio    # only top-ratio neurons fire per tick
```

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Prior (st08 sparsity0.5)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    plot_prior_bar(df_prior, ['sort','mazes'],'st08','sparsity0.5','Prior: sparsity 0.5','figures/03_prior.png')
else: print('no prior')

## Group 1: Main (ratio=0.5, 5 seeds)

### sort — topk=0.5, 5 seeds

In [ ]:
module, base = BASE_CONFIGS['sort']
exps_main_sort = [
    Experiment(f'sort_sparsity0p5_s{s}', 'sort', module,
              {**base, 'seed': s, 'topk_neurons': 0.5})
    for s in range(5)]
print(f'{len(exps_main_sort)} runs')

### mazes — topk=0.5, 5 seeds

In [ ]:
module, base = BASE_CONFIGS['mazes']
exps_main_mazes = [
    Experiment(f'mazes_sparsity0p5_s{s}', 'mazes', module,
              {**base, 'seed': s, 'topk_neurons': 0.5})
    for s in range(5)]
print(f'{len(exps_main_mazes)} runs')

## Group 2: Ratio Sweep

### ratio=0.1 — very sparse (10% active)

3 seeds × 2 tasks = 6 runs.

In [ ]:
exps_swp_0p1 = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(3):
        exps_swp_0p1.append(Experiment(
            f'{task}_swp_r0p1_s{s}', task, module,
            {**base, 'seed': s, 'topk_neurons': 0.1}))  # SWEEP
print(f'{len(exps_swp_0p1)} runs')

### ratio=0.25 — sparse

3 seeds × 2 tasks = 6 runs.

In [ ]:
exps_swp_0p25 = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(3):
        exps_swp_0p25.append(Experiment(
            f'{task}_swp_r0p25_s{s}', task, module,
            {**base, 'seed': s, 'topk_neurons': 0.25}))  # SWEEP
print(f'{len(exps_swp_0p25)} runs')

### ratio=0.5 — default

3 seeds × 2 tasks = 6 runs.

In [ ]:
exps_swp_0p5 = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(3):
        exps_swp_0p5.append(Experiment(
            f'{task}_swp_r0p5_s{s}', task, module,
            {**base, 'seed': s, 'topk_neurons': 0.5}))  # SWEEP
print(f'{len(exps_swp_0p5)} runs')

### ratio=0.75 — mild

3 seeds × 2 tasks = 6 runs.

In [ ]:
exps_swp_0p75 = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(3):
        exps_swp_0p75.append(Experiment(
            f'{task}_swp_r0p75_s{s}', task, module,
            {**base, 'seed': s, 'topk_neurons': 0.75}))  # SWEEP
print(f'{len(exps_swp_0p75)} runs')

### ratio=0.9 — almost dense

3 seeds × 2 tasks = 6 runs.

In [ ]:
exps_swp_0p9 = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(3):
        exps_swp_0p9.append(Experiment(
            f'{task}_swp_r0p9_s{s}', task, module,
            {**base, 'seed': s, 'topk_neurons': 0.9}))  # SWEEP
print(f'{len(exps_swp_0p9)} runs')

## Run + Analyze

In [ ]:
exps = (exps_main_sort + exps_main_mazes
        + sum((v for k,v in list(globals().items()) if k.startswith('exps_swp_')), []))
print(f'Total: {len(exps)} experiments')
run_all(exps, gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/03_sparsity')

In [ ]:
status('logs/deep/03_sparsity')

In [ ]:
df = collect('logs/deep/03_sparsity')
if not df.empty:
    import re
    df_main = df[df.name.str.contains('sparsity0p5_s') & ~df.name.str.contains('swp')]
    if not df_main.empty:
        plot_delta_bars(df_main, 'Sparsity(0.5) (5 seeds)', 'figures/03_main.png')
        print(significance_test(df_main).to_string(index=False))
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['ratio'] = df_sw.name.str.extract(r'r([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'ratio', 'task', 'Ratio sweep', 'figures/03_sweep.png')
else: print('No results yet.')